In [ ]:
# Memory safety: cap this kernel to the RAM free right now so an out-of-memory
# feature build fails with a clean MemoryError instead of crashing VS Code /
# thrashing swap. This notebook builds cross-row features (lags/rolling), which
# can't be row-batched, so the guard is the protection here.
import os, sys
sys.path.insert(0, os.path.abspath("../../../Generic-Parallel-Compute-Helper/")) ; from memory_compute import *
install_memory_guard()


In [1]:
import os
import sys
import numpy as np
import pandas as pd

sys.path.append(os.path.abspath("../../")) ; from EPF import variables

In [ ]:
df = read_parquet_float32("../1_Dataset/Processed_data/1_dispatch_price.parquet")

df_core_columns = df.columns
df_base = df
df_base[:10]

,nsw_price,qld_price,sa_price,vic_price
Date,,,,
2018-01-01 00:05:00,90.21000,80.27029,103.57857,90.60964
2018-01-01 00:10:00,95.54066,85.10007,111.81715,96.73685
2018-01-01 00:15:00,96.29387,85.10000,113.70290,97.34439
2018-01-01 00:20:00,92.50000,81.72834,106.75178,92.82445
2018-01-01 00:25:00,92.49990,81.70893,108.53188,92.83313
2018-01-01 00:30:00,84.09234,73.73003,98.65979,84.38901
2018-01-01 00:35:00,92.44987,81.08865,105.00005,91.34952
2018-01-01 00:40:00,95.79015,84.59996,109.10280,93.91492
2018-01-01 00:45:00,90.99791,79.50082,105.00005,89.81217


In [3]:
def _add_arcsinh_price_lags(df: pd.DataFrame) -> pd.DataFrame:
    """
    arcsinh-transformed price lags.
    arcsinh handles negatives and compresses extreme spikes,
    giving the model a better-scaled view of price history.
    Returns only the new columns to avoid copying the full base frame.
    """
    scale = 100
    new_cols = {}
    for col in df_core_columns:
        ap = np.arcsinh(df[col] / scale).rename("_ap")
        for lag in [1, 2, 4, 12, 48, 96, 336, 335, 337]:
            new_cols[f"{col}_asinh_lag_{lag}"] = ap.shift(lag).astype(np.float32)
        new_cols[f"{col}_asinh_rmean_48"]  = ap.rolling(48).mean().astype(np.float32)
        new_cols[f"{col}_asinh_rmean_336"] = ap.rolling(336).mean().astype(np.float32)
    return pd.DataFrame(new_cols, index=df.index)

new_df = _add_arcsinh_price_lags(df_base)

df = pd.concat([df, new_df], axis=1)

new_df[:10]



,nsw_price_asinh_lag_1,nsw_price_asinh_lag_2,nsw_price_asinh_lag_4,nsw_price_asinh_lag_12,nsw_price_asinh_lag_48,nsw_price_asinh_lag_96,nsw_price_asinh_lag_336,nsw_price_asinh_lag_335,nsw_price_asinh_lag_337,nsw_price_asinh_rmean_48,...,vic_price_asinh_lag_2,vic_price_asinh_lag_4,vic_price_asinh_lag_12,vic_price_asinh_lag_48,vic_price_asinh_lag_96,vic_price_asinh_lag_336,vic_price_asinh_lag_335,vic_price_asinh_lag_337,vic_price_asinh_rmean_48,vic_price_asinh_rmean_336
Date,,,,,,,,,,,,,,,,,,,,,
2018-01-01 00:05:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:10:00,0.810427,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:15:00,0.849487,0.810427,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.813392,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:20:00,0.854923,0.849487,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.858110,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:25:00,0.827334,0.854923,0.810427,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.862470,0.813392,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:30:00,0.827333,0.827334,0.849487,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.829714,0.858110,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:35:00,0.764306,0.827333,0.854923,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.829778,0.862470,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:40:00,0.826966,0.764306,0.827334,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.766575,0.829714,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:45:00,0.851290,0.826966,0.827333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.818864,0.829778,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
def _add_time_features(df: pd.DataFrame) -> pd.DataFrame:

    import holidays

    _NSW_HOLIDAYS = holidays.Australia(state="NSW", years=range(2018, 2026))

    idx = df.index
    new_cols = {}

    # Raw calendar components
    new_cols["hour"]        = idx.hour.astype(np.int8)
    new_cols["dayofweek"]   = idx.dayofweek.astype(np.int8)
    new_cols["month"]       = idx.month.astype(np.int8)
    new_cols["dayofyear"]   = idx.day_of_year.astype(np.int16)

    # Cyclical (sin/cos) encodings so the model sees periodicity
    new_cols["hour_sin"]    = np.sin(2 * np.pi * idx.hour / 24).astype(np.float32)
    new_cols["hour_cos"]    = np.cos(2 * np.pi * idx.hour / 24).astype(np.float32)
    new_cols["dow_sin"]     = np.sin(2 * np.pi * idx.dayofweek / 7).astype(np.float32)
    new_cols["dow_cos"]     = np.cos(2 * np.pi * idx.dayofweek / 7).astype(np.float32)
    new_cols["month_sin"]   = np.sin(2 * np.pi * (idx.month - 1) / 12).astype(np.float32)
    new_cols["month_cos"]   = np.cos(2 * np.pi * (idx.month - 1) / 12).astype(np.float32)

    # Binary flags
    is_weekend = (idx.dayofweek >= 5).astype(np.float32)
    is_holiday = np.array(
        [d.date() in _NSW_HOLIDAYS for d in idx], dtype=np.float32
    )
    new_cols["is_weekend"]  = is_weekend
    new_cols["is_holiday"]  = is_holiday
    # Peak (17–20 h) and off-peak (< 7 h or ≥ 21 h) periods
    new_cols["is_peak"]     = ((idx.hour >= 17) & (idx.hour <= 20)).astype(np.float32)
    new_cols["is_shoulder"] = ((idx.hour >= 7)  & (idx.hour < 17)).astype(np.float32)
    new_cols["is_off_peak"] = ((idx.hour < 7)   | (idx.hour >= 21)).astype(np.float32)

    # Combined weekend/holiday flag — demand behaviour is quite different
    new_cols["is_offday"]   = np.maximum(is_weekend, is_holiday).astype(np.float32)

    return pd.DataFrame(new_cols, index=idx)

new_df = _add_time_features(df_base)

df = pd.concat([df, new_df], axis=1)

new_df[:10]



,hour,dayofweek,month,dayofyear,hour_sin,hour_cos,dow_sin,dow_cos,month_sin,month_cos,is_weekend,is_holiday,is_peak,is_shoulder,is_off_peak,is_offday
Date,,,,,,,,,,,,,,,,
2018-01-01 00:05:00,0,0,1,1,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0
2018-01-01 00:10:00,0,0,1,1,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0
2018-01-01 00:15:00,0,0,1,1,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0
2018-01-01 00:20:00,0,0,1,1,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0
2018-01-01 00:25:00,0,0,1,1,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0
2018-01-01 00:30:00,0,0,1,1,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0
2018-01-01 00:35:00,0,0,1,1,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0
2018-01-01 00:40:00,0,0,1,1,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0
2018-01-01 00:45:00,0,0,1,1,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0


In [5]:
def _add_lag_features(df: pd.DataFrame) -> pd.DataFrame:
    LAG_INTERVALS = sorted(set([
        1, 2, 3, 4, 6, 8, 12, 24,
        48, 49, 50, 51, 52,
        96, 97, 98,
        # 3-day and 4-day same-period anchors — captures weekly envelope effects
        # without redundancy with the weekly (336) lag.
        144, 143, 145,
        192, 191, 193,
        336, 335, 337,
        672, 671, 673,
        # One full week at 5-min granularity (7*24*60/5 = 2016) — seasonal-naive baseline predictor.
        2016,
    ] + list(range(26, 72, 2))))  # even lags covering 13h–35h window

    new_cols = {}
    for col in df_core_columns:
        for lag in LAG_INTERVALS:
            new_cols[f"{col}_lag_{lag}"] = df[col].shift(lag).astype(np.float32)

    return pd.DataFrame(new_cols, index=df.index)

new_df = _add_lag_features(df_base)

df = pd.concat([df, new_df], axis=1)

new_df[:10]



,nsw_price_lag_1,nsw_price_lag_2,nsw_price_lag_3,nsw_price_lag_4,nsw_price_lag_6,nsw_price_lag_8,nsw_price_lag_12,nsw_price_lag_24,nsw_price_lag_26,nsw_price_lag_28,...,vic_price_lag_191,vic_price_lag_192,vic_price_lag_193,vic_price_lag_335,vic_price_lag_336,vic_price_lag_337,vic_price_lag_671,vic_price_lag_672,vic_price_lag_673,vic_price_lag_2016
Date,,,,,,,,,,,,,,,,,,,,,
2018-01-01 00:05:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:10:00,90.209999,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:15:00,95.540657,90.209999,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:20:00,96.293869,95.540657,90.209999,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:25:00,92.500000,96.293869,95.540657,90.209999,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:30:00,92.499901,92.500000,96.293869,95.540657,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:35:00,84.092339,92.499901,92.500000,96.293869,90.209999,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:40:00,92.449867,84.092339,92.499901,92.500000,95.540657,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:45:00,95.790154,92.449867,84.092339,92.499901,96.293869,90.209999,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
def _add_long_range_features(df: pd.DataFrame) -> pd.DataFrame:

    new_cols = {}

    for col in df_core_columns:

        ap   = np.arcsinh(df[col] / 100)
        BASE = 17_532

        # --- Annual price lags (same period last year ± spread) ---
        for offset in range(-2, 2 + 1):
            lag = BASE + offset
            new_cols[f"{col}_lag_annual_{'+' if offset >= 0 else ''}{offset}"] = (
                df[col].shift(lag).astype(np.float32)
            )
            new_cols[f"{col}_asinh_lag_annual_{'+' if offset >= 0 else ''}{offset}"] = (
                ap.shift(lag).astype(np.float32)
            )

        p_annual = df[col].shift(BASE)
        new_cols[f"{col}_annual_rmean_96"]  = p_annual.rolling(96, min_periods=24).mean().astype(np.float32)
        new_cols[f"{col}_annual_rmax_96"]   = p_annual.rolling(96, min_periods=24).max().astype(np.float32)
        new_cols[f"{col}_annual_rstd_96"]   = p_annual.rolling(96, min_periods=24).std().astype(np.float32)

        # Spike count over the same week last year
        new_cols[f"{col}_annual_spike_96"]  = (p_annual >= 300).rolling(96, min_periods=24).sum().astype(np.float32)

        # Year-on-year price change (current vs same period last year)
        new_cols[f"{col}_yoy_change"]  = (df[col] - df[col].shift(BASE)).astype(np.float32)
        new_cols[f"{col}_yoy_ratio"]   = (df[col] / (df[col].shift(BASE).abs() + 1)).clip(0, 20).astype(np.float32)

        # Note: 2-week and 6-week rolling stats ({col}_rmean_672, _rmax_672, _rmin_672,
        # _rstd_672, _rmean_2016) are generated by _add_rolling_features to avoid duplication.

    return pd.DataFrame(new_cols, index=df.index)

new_df = _add_long_range_features(df_base)

df = pd.concat([df, new_df], axis=1)

new_df[:10]



,nsw_price_lag_annual_-2,nsw_price_asinh_lag_annual_-2,nsw_price_lag_annual_-1,nsw_price_asinh_lag_annual_-1,nsw_price_lag_annual_+0,nsw_price_asinh_lag_annual_+0,nsw_price_lag_annual_+1,nsw_price_asinh_lag_annual_+1,nsw_price_lag_annual_+2,nsw_price_asinh_lag_annual_+2,...,vic_price_lag_annual_+1,vic_price_asinh_lag_annual_+1,vic_price_lag_annual_+2,vic_price_asinh_lag_annual_+2,vic_price_annual_rmean_96,vic_price_annual_rmax_96,vic_price_annual_rstd_96,vic_price_annual_spike_96,vic_price_yoy_change,vic_price_yoy_ratio
Date,,,,,,,,,,,,,,,,,,,,,
2018-01-01 00:05:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:10:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:15:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:20:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:25:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:35:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:40:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:45:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
def _add_rolling_features(df: pd.DataFrame) -> pd.DataFrame:
    """Rolling statistics computed on price available at each timestamp."""
    ROLLING_WINDOWS = [4, 8, 24, 48, 96, 336, 672, 2016]

    new_cols = {}
    for col in df_core_columns:
        for w in ROLLING_WINDOWS:
            min_p = max(1, w // 2)
            rolled = df[col].rolling(w, min_periods=min_p)
            new_cols[f"{col}_rmean_{w}"] = rolled.mean().astype(np.float32)
            new_cols[f"{col}_rstd_{w}"]  = rolled.std().astype(np.float32)
            new_cols[f"{col}_rmax_{w}"]  = rolled.max().astype(np.float32)
            new_cols[f"{col}_rmin_{w}"]  = rolled.min().astype(np.float32)  

    return pd.DataFrame(new_cols, index=df.index)

new_df = _add_rolling_features(df_base)

df = pd.concat([df, new_df], axis=1)

new_df[:10]



,nsw_price_rmean_4,nsw_price_rstd_4,nsw_price_rmax_4,nsw_price_rmin_4,nsw_price_rmean_8,nsw_price_rstd_8,nsw_price_rmax_8,nsw_price_rmin_8,nsw_price_rmean_24,nsw_price_rstd_24,...,vic_price_rmax_336,vic_price_rmin_336,vic_price_rmean_672,vic_price_rstd_672,vic_price_rmax_672,vic_price_rmin_672,vic_price_rmean_2016,vic_price_rstd_2016,vic_price_rmax_2016,vic_price_rmin_2016
Date,,,,,,,,,,,,,,,,,,,,,
2018-01-01 00:05:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:10:00,92.875328,3.769346,95.540657,90.209999,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:15:00,94.014847,3.316543,96.293869,90.209999,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:20:00,93.636131,2.811878,96.293869,90.209999,93.636131,2.811878,96.293869,90.209999,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:25:00,94.208611,1.996806,96.293869,92.499901,93.408890,2.487609,96.293869,90.209999,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:30:00,91.346527,5.156232,96.293869,84.092339,91.856125,4.406461,96.293869,84.092339,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:35:00,90.385529,4.195525,92.500000,84.092339,91.940948,4.028785,96.293869,84.092339,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:40:00,91.208061,4.994663,95.790154,84.092339,92.422096,3.970444,96.293869,84.092339,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:45:00,90.832565,4.921087,95.790154,84.092339,92.520584,3.917147,96.293869,84.092339,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
def _add_regime_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Capture price-spike and volatility regime signals.
    All windows look backward only, so there is no future leakage.
    Note: vol_48/vol_336 (rolling std) are already produced as rstd_48/rstd_336
    by _add_rolling_features and are not repeated here.
    Returns only the new columns to avoid copying the full base frame.
    """
    new_cols = {}

    for col in df_core_columns:

        # Was there a high-price spike in the last 24 h?
        new_cols[f"{col}_spike_flag_48"]    = (df[col].rolling(48).max()  > 300).astype(np.float32)
        # Was there a negative price in the last 24 h?
        new_cols[f"{col}_neg_flag_48"]      = (df[col].rolling(48).min()  < 0).astype(np.float32)
        # Quantile context (recent price level relative to weekly range)
        new_cols[f"{col}_q90_336"]    = df[col].rolling(336).quantile(0.90).astype(np.float32)
        new_cols[f"{col}_q10_336"]    = df[col].rolling(336).quantile(0.10).astype(np.float32)
        new_cols[f"{col}_pct_rank_48"] = (
            df[col].rolling(48).rank(pct=True)
        ).astype(np.float32)

    return pd.DataFrame(new_cols, index=df.index)

new_df = _add_regime_features(df_base)

df = pd.concat([df, new_df], axis=1)

new_df[:10]



,nsw_price_spike_flag_48,nsw_price_neg_flag_48,nsw_price_q90_336,nsw_price_q10_336,nsw_price_pct_rank_48,qld_price_spike_flag_48,qld_price_neg_flag_48,qld_price_q90_336,qld_price_q10_336,qld_price_pct_rank_48,sa_price_spike_flag_48,sa_price_neg_flag_48,sa_price_q90_336,sa_price_q10_336,sa_price_pct_rank_48,vic_price_spike_flag_48,vic_price_neg_flag_48,vic_price_q90_336,vic_price_q10_336,vic_price_pct_rank_48
Date,,,,,,,,,,,,,,,,,,,,
2018-01-01 00:05:00,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN
2018-01-01 00:10:00,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN
2018-01-01 00:15:00,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN
2018-01-01 00:20:00,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN
2018-01-01 00:25:00,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN
2018-01-01 00:30:00,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN
2018-01-01 00:35:00,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN
2018-01-01 00:40:00,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN
2018-01-01 00:45:00,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN


In [9]:
def _add_time_since_spike_features(df: pd.DataFrame) -> pd.DataFrame:

    # 30-min intervals between rows
    INTERVAL_H = 0.5
    # Maximum hours to report (cap at 2 weeks to avoid unbounded values early in series)
    MAX_HOURS  = 336.0  # 2 weeks

    new_cols = {}
    for col in df_core_columns:
        hours_since_by_thr = {}
        for threshold in [150, 300, 1000]:
            spike_flag = (df[col] >= threshold).astype(np.float32)

            positions       = pd.RangeIndex(len(df))
            last_spike_pos  = (
                pd.Series(np.where(spike_flag.values, positions, np.nan), index=df.index)
                .ffill()
                .fillna(-MAX_HOURS / INTERVAL_H)  # no prior spike seen → treat as max
            )
            intervals_since = (pd.Series(positions, index=df.index) - last_spike_pos).clip(upper=MAX_HOURS / INTERVAL_H)
            hours_since     = (intervals_since * INTERVAL_H).astype(np.float32)
            new_cols[f"{col}_hours_since_spike_{threshold}"] = hours_since
            hours_since_by_thr[threshold] = hours_since

            # Log-scale version reduces the numeric range for the tree learner
            new_cols[f"{col}_log1p_hours_since_spike_{threshold}"] = np.log1p(hours_since).astype(np.float32)

            # Binary flags: was there a spike at each lookback horizon?
            for lookback_h in [1, 6, 12, 24, 48, 168]:   # 1h, 6h, 12h, 24h, 48h, 1wk
                intervals = int(lookback_h / INTERVAL_H)
                new_cols[f"{col}_spike_{threshold}_last_{lookback_h}h"] = (
                    spike_flag.rolling(intervals, min_periods=1).max().astype(np.float32)
                )

        # Combined: hours since any of the three thresholds (summarises overall stress state)
        new_cols[f"{col}_hours_since_any_spike"] = pd.concat(
            [hours_since_by_thr[150], hours_since_by_thr[300], hours_since_by_thr[1000]], axis=1
        ).min(axis=1).astype(np.float32)

    return pd.DataFrame(new_cols, index=df.index)


new_df = _add_time_since_spike_features(df_base)

df = pd.concat([df, new_df], axis=1)

new_df[:10]



,nsw_price_hours_since_spike_150,nsw_price_log1p_hours_since_spike_150,nsw_price_spike_150_last_1h,nsw_price_spike_150_last_6h,nsw_price_spike_150_last_12h,nsw_price_spike_150_last_24h,nsw_price_spike_150_last_48h,nsw_price_spike_150_last_168h,nsw_price_hours_since_spike_300,nsw_price_log1p_hours_since_spike_300,...,vic_price_spike_300_last_168h,vic_price_hours_since_spike_1000,vic_price_log1p_hours_since_spike_1000,vic_price_spike_1000_last_1h,vic_price_spike_1000_last_6h,vic_price_spike_1000_last_12h,vic_price_spike_1000_last_24h,vic_price_spike_1000_last_48h,vic_price_spike_1000_last_168h,vic_price_hours_since_any_spike
Date,,,,,,,,,,,,,,,,,,,,,
2018-01-01 00:05:00,336.0,5.820083,0.0,0.0,0.0,0.0,0.0,0.0,336.0,5.820083,...,0.0,336.0,5.820083,0.0,0.0,0.0,0.0,0.0,0.0,336.0
2018-01-01 00:10:00,336.0,5.820083,0.0,0.0,0.0,0.0,0.0,0.0,336.0,5.820083,...,0.0,336.0,5.820083,0.0,0.0,0.0,0.0,0.0,0.0,336.0
2018-01-01 00:15:00,336.0,5.820083,0.0,0.0,0.0,0.0,0.0,0.0,336.0,5.820083,...,0.0,336.0,5.820083,0.0,0.0,0.0,0.0,0.0,0.0,336.0
2018-01-01 00:20:00,336.0,5.820083,0.0,0.0,0.0,0.0,0.0,0.0,336.0,5.820083,...,0.0,336.0,5.820083,0.0,0.0,0.0,0.0,0.0,0.0,336.0
2018-01-01 00:25:00,336.0,5.820083,0.0,0.0,0.0,0.0,0.0,0.0,336.0,5.820083,...,0.0,336.0,5.820083,0.0,0.0,0.0,0.0,0.0,0.0,336.0
2018-01-01 00:30:00,336.0,5.820083,0.0,0.0,0.0,0.0,0.0,0.0,336.0,5.820083,...,0.0,336.0,5.820083,0.0,0.0,0.0,0.0,0.0,0.0,336.0
2018-01-01 00:35:00,336.0,5.820083,0.0,0.0,0.0,0.0,0.0,0.0,336.0,5.820083,...,0.0,336.0,5.820083,0.0,0.0,0.0,0.0,0.0,0.0,336.0
2018-01-01 00:40:00,336.0,5.820083,0.0,0.0,0.0,0.0,0.0,0.0,336.0,5.820083,...,0.0,336.0,5.820083,0.0,0.0,0.0,0.0,0.0,0.0,336.0
2018-01-01 00:45:00,336.0,5.820083,0.0,0.0,0.0,0.0,0.0,0.0,336.0,5.820083,...,0.0,336.0,5.820083,0.0,0.0,0.0,0.0,0.0,0.0,336.0


In [10]:
def _add_spike_predictors(df: pd.DataFrame) -> pd.DataFrame:
    """
    Price momentum and spike-history features.
    All look-back only — zero leakage.
    Note: pct_rank_48 is already produced by _add_regime_features and is not repeated here.
    Returns only the new columns to avoid copying the full base frame.
    """

    new_cols = {}

    for col in df_core_columns:

        # Price momentum — how fast prices are moving right now
        new_cols[f"{col}_mom_4"]  = df[col].diff(4).astype(np.float32)    # 2h
        new_cols[f"{col}_mom_12"] = df[col].diff(12).astype(np.float32)   # 6h
        new_cols[f"{col}_mom_48"] = df[col].diff(48).astype(np.float32)   # 24h

        # Price acceleration (second derivative) — is the current spike accelerating?
        new_cols[f"{col}_accel_4"]  = df[col].diff(4).diff(4).clip(-2000, 2000).astype(np.float32)
        new_cols[f"{col}_accel_12"] = df[col].diff(12).diff(12).clip(-2000, 2000).astype(np.float32)

        # Spike and negative price counts in recent history
        new_cols[f"{col}_spike_count_48"]  = (df[col] >= 300).rolling(48).sum().astype(np.float32)
        new_cols[f"{col}_spike_count_336"] = (df[col] >= 300).rolling(336).sum().astype(np.float32)
        new_cols[f"{col}_neg_count_48"]    = (df[col] < 0).rolling(48).sum().astype(np.float32)

        # Spike intensity: cumulative spike energy in last 24h (not just count)
        SPIKE_THR = 150.0
        spike_excess = (df[col] - SPIKE_THR).clip(lower=0)
        new_cols[f"{col}_spike_intensity_48"]  = spike_excess.rolling(48).sum().astype(np.float32)
        new_cols[f"{col}_spike_intensity_336"] = spike_excess.rolling(336).sum().astype(np.float32)

        # Price percentile rank over weekly window
        new_cols[f"{col}_pct_rank_336"] = df[col].rolling(336).rank(pct=True).astype(np.float32)

    return pd.DataFrame(new_cols, index=df.index)

new_df = _add_spike_predictors(df_base)

df = pd.concat([df, new_df], axis=1)

new_df[:10]



,nsw_price_mom_4,nsw_price_mom_12,nsw_price_mom_48,nsw_price_accel_4,nsw_price_accel_12,nsw_price_spike_count_48,nsw_price_spike_count_336,nsw_price_neg_count_48,nsw_price_spike_intensity_48,nsw_price_spike_intensity_336,...,vic_price_mom_12,vic_price_mom_48,vic_price_accel_4,vic_price_accel_12,vic_price_spike_count_48,vic_price_spike_count_336,vic_price_neg_count_48,vic_price_spike_intensity_48,vic_price_spike_intensity_336,vic_price_pct_rank_336
Date,,,,,,,,,,,,,,,,,,,,,
2018-01-01 00:05:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:10:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:15:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:20:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:25:00,2.28990,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:30:00,-11.44832,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:35:00,-3.84400,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:40:00,3.29015,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:45:00,-1.50199,NaN,NaN,-3.79189,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,-5.24445,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
def _add_region_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Pairwise spread features for all NEM region prices (NSW, QLD, VIC, SA).
    Captures interconnector pressure from any direction — useful for predicting any state.

    Lags, rolling stats, arcsinh lags, and spike counts are handled by their dedicated
    functions (_add_lag_features, _add_rolling_features, _add_arcsinh_price_lags,
    _add_spike_predictors) and are not repeated here.
    Returns only the new columns to avoid copying the full base frame.
    """

    new_cols: dict = {}

    for col in df_core_columns:
        # Pairwise spreads vs every other region — captures interconnector pressure
        for other_col in df_core_columns:
            if other_col == col:
                continue
            spread = (df[col] - df[other_col]).astype(np.float32)
            new_cols[f"{col}_vs_{other_col}_spread"]      = spread
            new_cols[f"{col}_vs_{other_col}_spread_lag1"] = spread.shift(1).astype(np.float32)

    # Multi-region spike co-occurrence: all regions elevated simultaneously
    regions_present = [c for c in df_core_columns if c in df.columns]
    if len(regions_present) >= 2:
        flags = pd.concat([(df[c] >= 150).astype(np.float32) for c in regions_present], axis=1)
        new_cols["multi_region_spike"] = flags.min(axis=1)  # 1 only if ALL elevated
        new_cols["region_spike_count"] = flags.sum(axis=1).astype(np.float32)  # 0–4

    return pd.DataFrame(new_cols, index=df.index)

new_df = _add_region_features(df_base)

df = pd.concat([df, new_df], axis=1)

new_df[:10]



,nsw_price_vs_qld_price_spread,nsw_price_vs_qld_price_spread_lag1,nsw_price_vs_sa_price_spread,nsw_price_vs_sa_price_spread_lag1,nsw_price_vs_vic_price_spread,nsw_price_vs_vic_price_spread_lag1,qld_price_vs_nsw_price_spread,qld_price_vs_nsw_price_spread_lag1,qld_price_vs_sa_price_spread,qld_price_vs_sa_price_spread_lag1,...,sa_price_vs_vic_price_spread,sa_price_vs_vic_price_spread_lag1,vic_price_vs_nsw_price_spread,vic_price_vs_nsw_price_spread_lag1,vic_price_vs_qld_price_spread,vic_price_vs_qld_price_spread_lag1,vic_price_vs_sa_price_spread,vic_price_vs_sa_price_spread_lag1,multi_region_spike,region_spike_count
Date,,,,,,,,,,,,,,,,,,,,,
2018-01-01 00:05:00,9.93971,NaN,-13.368570,NaN,-0.39964,NaN,-9.93971,NaN,-23.308281,NaN,...,12.968930,NaN,0.39964,NaN,10.33935,NaN,-12.968930,NaN,0.0,0.0
2018-01-01 00:10:00,10.44059,9.93971,-16.276489,-13.368570,-1.19619,-0.39964,-10.44059,-9.93971,-26.717079,-23.308281,...,15.080300,12.968930,1.19619,0.39964,11.63678,10.33935,-15.080300,-12.968930,0.0,0.0
2018-01-01 00:15:00,11.19387,10.44059,-17.409031,-16.276489,-1.05052,-1.19619,-11.19387,-10.44059,-28.602900,-26.717079,...,16.358509,15.080300,1.05052,1.19619,12.24439,11.63678,-16.358509,-15.080300,0.0,0.0
2018-01-01 00:20:00,10.77166,11.19387,-14.251780,-17.409031,-0.32445,-1.05052,-10.77166,-11.19387,-25.023439,-28.602900,...,13.927330,16.358509,0.32445,1.05052,11.09611,12.24439,-13.927330,-16.358509,0.0,0.0
2018-01-01 00:25:00,10.79097,10.77166,-16.031981,-14.251780,-0.33323,-0.32445,-10.79097,-10.77166,-26.822950,-25.023439,...,15.698750,13.927330,0.33323,0.32445,11.12420,11.09611,-15.698750,-13.927330,0.0,0.0
2018-01-01 00:30:00,10.36231,10.79097,-14.567450,-16.031981,-0.29667,-0.33323,-10.36231,-10.79097,-24.929760,-26.822950,...,14.270780,15.698750,0.29667,0.33323,10.65898,11.12420,-14.270780,-15.698750,0.0,0.0
2018-01-01 00:35:00,11.36122,10.36231,-12.550180,-14.567450,1.10035,-0.29667,-11.36122,-10.36231,-23.911400,-24.929760,...,13.650530,14.270780,-1.10035,0.29667,10.26087,10.65898,-13.650530,-14.270780,0.0,0.0
2018-01-01 00:40:00,11.19019,11.36122,-13.312650,-12.550180,1.87523,1.10035,-11.19019,-11.36122,-24.502840,-23.911400,...,15.187880,13.650530,-1.87523,-1.10035,9.31496,10.26087,-15.187880,-13.650530,0.0,0.0
2018-01-01 00:45:00,11.49709,11.19019,-14.002140,-13.312650,1.18574,1.87523,-11.49709,-11.19019,-25.499229,-24.502840,...,15.187880,15.187880,-1.18574,-1.87523,10.31135,9.31496,-15.187880,-15.187880,0.0,0.0


In [12]:
def _add_cross_feats(df: pd.DataFrame) -> pd.DataFrame:
    doy = df.index.day_of_year.astype(np.float32)
    new_cols = {}
    new_cols["doy_sin"] = np.sin(2 * np.pi * doy / 365.25).astype(np.float32)
    new_cols["doy_cos"] = np.cos(2 * np.pi * doy / 365.25).astype(np.float32)
    if "sa_price" in df.columns and "nsw_price" in df.columns:
        new_cols["sa_spread_live"] = np.arcsinh(
            (df["sa_price"].fillna(0.0).values - df["nsw_price"].fillna(0.0).values) / 100.0
        ).clip(-10, 10).astype(np.float32)
    return pd.DataFrame(new_cols, index=df.index)

new_df = _add_cross_feats(df_base)
df = pd.concat([df, new_df], axis=1)
new_df[:10]


,doy_sin,doy_cos,sa_spread_live
Date,,,
2018-01-01 00:05:00,0.017202,0.999852,0.133291
2018-01-01 00:10:00,0.017202,0.999852,0.162055
2018-01-01 00:15:00,0.017202,0.999852,0.173223
2018-01-01 00:20:00,0.017202,0.999852,0.142040
2018-01-01 00:25:00,0.017202,0.999852,0.159641
2018-01-01 00:30:00,0.017202,0.999852,0.145164
2018-01-01 00:35:00,0.017202,0.999852,0.125175
2018-01-01 00:40:00,0.017202,0.999852,0.132736
2018-01-01 00:45:00,0.017202,0.999852,0.139568


In [13]:
# Retain core columns: current dispatch prices are known at origin t, so keeping
# them as features is leakage-free (targets are strictly future, >= +30 min).
print("Total features:", df.shape[1])

df.to_parquet("../2_Features_build/Feature_data/1_dispatch_price.parquet")
df.shape

Total features: 645


(893664, 645)

In [ ]:
# Free this kernel's memory so the next notebook has RAM to work with
# (clears data variables + returns freed heap to the OS).
release_memory()
